In [ ]:
"""
inference.py

Loads the trained U-Net and generates high-resolution LST
predictions from processed input patches.

Input:
    X = (N, 10, 256, 256)

Output:
    predictions = (N, 1, 256, 256)
"""

from pathlib import Path

import numpy as np
import torch

from unet_model import UNet


# ============================================================
# Paths
# ============================================================

DATA_DIR = Path(
    "/content/drive/MyDrive/SISTER/data"
)

PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"

PREDICTION_DIR = DATA_DIR / "predictions"


# ============================================================
# Configuration
# ============================================================

BATCH_SIZE = 2


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# Load processed input data
# ============================================================

def load_input_data():
    """
    Load the preprocessed U-Net inputs.

    Expected shape:

        X = (N, 10, 256, 256)
    """

    input_path = (
        PROCESSED_DIR / "X.npy"
    )

    if not input_path.exists():

        raise FileNotFoundError(
            f"Processed input data not found:\n"
            f"{input_path}"
        )

    X = np.load(
        input_path
    )

    if X.ndim != 4:

        raise ValueError(
            f"X must be 4D, got {X.ndim}D."
        )

    if X.shape[1] != 10:

        raise ValueError(
            f"Expected 10 input channels, "
            f"got {X.shape[1]}."
        )

    print(
        "Input shape:",
        X.shape
    )

    return X


# ============================================================
# Load trained U-Net
# ============================================================

def load_model():
    """
    Load the best trained U-Net.
    """

    model = UNet(
        in_channels=10,
        out_channels=1
    ).to(
        DEVICE
    )

    model_path = (
        MODEL_DIR
        /
        "best_unet_model.pth"
    )

    if not model_path.exists():

        raise FileNotFoundError(
            f"Trained model not found:\n"
            f"{model_path}"
        )

    model.load_state_dict(
        torch.load(
            model_path,
            map_location=DEVICE
        )
    )

    model.eval()

    print(
        "Loaded model:",
        model_path
    )

    return model


# ============================================================
# Generate predictions
# ============================================================

def generate_predictions(
    model,
    X,
    batch_size=BATCH_SIZE
):
    """
    Generate LST predictions for every input patch.

    Returns:

        predictions:
            (N, 1, 256, 256)
    """

    predictions = []


    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X)
            )


            # ------------------------------------------------
            # Convert batch to tensor
            # ------------------------------------------------

            inputs = torch.tensor(
                X[start:end],
                dtype=torch.float32,
                device=DEVICE
            )


            # ------------------------------------------------
            # U-Net prediction
            # ------------------------------------------------

            outputs = model(
                inputs
            )


            # ------------------------------------------------
            # Move predictions to CPU
            # ------------------------------------------------

            predictions.append(
                outputs.cpu().numpy()
            )


    predictions = np.concatenate(
        predictions,
        axis=0
    )


    return predictions


# ============================================================
# Save predictions
# ============================================================

def save_predictions(
    predictions
):
    """
    Save predicted LST patches as a NumPy array.
    """

    PREDICTION_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    prediction_path = (
        PREDICTION_DIR
        /
        "predicted_lst.npy"
    )

    np.save(
        prediction_path,
        predictions
    )

    print(
        "\nPredictions saved to:"
    )

    print(
        prediction_path
    )

    return prediction_path


# ============================================================
# Run inference
# ============================================================

def run_inference():

    print(
        "Using device:",
        DEVICE
    )


    # --------------------------------------------------------
    # Load input data
    # --------------------------------------------------------

    X = load_input_data()


    # --------------------------------------------------------
    # Load trained model
    # --------------------------------------------------------

    model = load_model()


    # --------------------------------------------------------
    # Generate predictions
    # --------------------------------------------------------

    predictions = generate_predictions(
        model,
        X
    )


    # --------------------------------------------------------
    # Check output shape
    # --------------------------------------------------------

    expected_shape = (
        len(X),
        1,
        X.shape[2],
        X.shape[3]
    )

    if predictions.shape != expected_shape:

        raise ValueError(
            f"Unexpected prediction shape: "
            f"{predictions.shape}. "
            f"Expected: {expected_shape}."
        )


    print(
        "\nPrediction shape:",
        predictions.shape
    )


    # --------------------------------------------------------
    # Save predictions
    # --------------------------------------------------------

    save_predictions(
        predictions
    )


    return predictions


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    run_inference()